# Code 8.2.2: Parameter count, leak test, and GPT-2's released weights

Builds a GPT-2 small configuration, compares its parameter count with the formula, runs the causal leak test of Section 6.4, then copies in GPT-2 small's released weights and compares the logits with Hugging Face's `GPT2LMHeadModel`. It imports Code 8.2.1.


In [ ]:
from pathlib import Path

# Code 8.2.1, saved as gpt.py so `from gpt import ...` works in this notebook.
_GPT_PY = 'import math\nfrom dataclasses import dataclass\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n@dataclass\nclass GPTConfig:\n    V: int = 50257          # vocabulary size\n    n_max: int = 1024       # maximum context length\n    N: int = 12             # number of blocks\n    d: int = 768            # model width\n    h: int = 12             # heads\n    dropout: float = 0.0\n\nclass CausalSelfAttention(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.qkv = nn.Linear(cfg.d, 3 * cfg.d)          # W_Q, W_K, W_V in one matrix\n        self.proj = nn.Linear(cfg.d, cfg.d)             # W_O\n        self.h, self.dropout = cfg.h, cfg.dropout\n    def forward(self, x):\n        B, n, d = x.shape\n        q, k, v = self.qkv(x).split(d, dim=-1)\n        q, k, v = (t.view(B, n, self.h, d // self.h).transpose(1, 2) for t in (q, k, v))\n        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,   # the causal mask of Section 6.4\n                                           dropout_p=self.dropout if self.training else 0.0)\n        return self.proj(y.transpose(1, 2).reshape(B, n, d))\n\nclass MLP(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.fc = nn.Linear(cfg.d, 4 * cfg.d)\n        self.proj = nn.Linear(4 * cfg.d, cfg.d)\n    def forward(self, x):\n        return self.proj(F.gelu(self.fc(x), approximate="tanh"))   # GPT-2 uses the tanh form of GELU\n\nclass Block(nn.Module):                                 # pre-norm decoder block, no cross-attention\n    def __init__(self, cfg):\n        super().__init__()\n        self.ln1, self.attn = nn.LayerNorm(cfg.d), CausalSelfAttention(cfg)\n        self.ln2, self.mlp = nn.LayerNorm(cfg.d), MLP(cfg)\n        self.drop = nn.Dropout(cfg.dropout)\n    def forward(self, x):\n        x = x + self.drop(self.attn(self.ln1(x)))\n        return x + self.drop(self.mlp(self.ln2(x)))\n\nclass GPT(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        self.tok = nn.Embedding(cfg.V, cfg.d)\n        self.pos = nn.Embedding(cfg.n_max, cfg.d)       # learned absolute positions\n        self.drop = nn.Dropout(cfg.dropout)\n        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg.N))\n        self.ln_f = nn.LayerNorm(cfg.d)                 # final LayerNorm after the last block\n        self.apply(self._init)\n        for name, p in self.named_parameters():         # residual branches: std / sqrt(2N)\n            if name.endswith("proj.weight"):\n                nn.init.normal_(p, std=0.02 / math.sqrt(2 * cfg.N))\n    def _init(self, m):\n        if isinstance(m, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(m.weight, std=0.02)\n        if isinstance(m, nn.Linear) and m.bias is not None:\n            nn.init.zeros_(m.bias)\n    def forward(self, idx, targets=None):\n        n = idx.size(1)\n        assert n <= self.cfg.n_max\n        x = self.drop(self.tok(idx) + self.pos(torch.arange(n, device=idx.device)))\n        for block in self.blocks:\n            x = block(x)\n        logits = self.ln_f(x) @ self.tok.weight.T       # output projection tied to the embedding\n        if targets is None:\n            return logits\n        return logits, F.cross_entropy(logits.flatten(0, 1), targets.flatten())\n'
Path('gpt.py').write_text(_GPT_PY)


In [ ]:
import torch
from gpt import GPT, GPTConfig
from transformers import GPT2LMHeadModel

cfg = GPTConfig()                                       # GPT-2 small
torch.manual_seed(0)
model = GPT(cfg).eval()
N, d, V, n_max = cfg.N, cfg.d, cfg.V, cfg.n_max
print(f"{12 * N * d**2 + V * d + n_max * d:,}")         # formula: weights only
print(f"{sum(p.numel() for p in model.parameters()):,}")   # with biases and LayerNorms

# Causal leak test (Section 6.4): changing tokens from position 10 on must not change outputs 1-9
x = torch.randint(0, V, (1, 16))
x2 = x.clone(); x2[0, 9:] = torch.randint(0, V, (7,))
with torch.no_grad():
    print(torch.allclose(model(x)[0, :9], model(x2)[0, :9], atol=1e-5))

# Load GPT-2 small's released weights and compare with Hugging Face
hf = GPT2LMHeadModel.from_pretrained("gpt2").eval()
sd = hf.state_dict()
names = {"attn.c_attn": "attn.qkv", "attn.c_proj": "attn.proj", "mlp.c_fc": "mlp.fc",
         "mlp.c_proj": "mlp.proj", "ln_1": "ln1", "ln_2": "ln2"}
with torch.no_grad():
    model.tok.weight.copy_(sd["transformer.wte.weight"])
    model.pos.weight.copy_(sd["transformer.wpe.weight"])
    model.ln_f.load_state_dict({"weight": sd["transformer.ln_f.weight"], "bias": sd["transformer.ln_f.bias"]})
    for i, block in enumerate(model.blocks):
        for hf_name, my_name in names.items():
            w, b = sd[f"transformer.h.{i}.{hf_name}.weight"], sd[f"transformer.h.{i}.{hf_name}.bias"]
            mod = block.get_submodule(my_name)
            mod.weight.copy_(w if hf_name.startswith("ln") else w.T)   # HF's Conv1D stores W transposed
            mod.bias.copy_(b)
    torch.manual_seed(1)
    x = torch.randint(0, V, (2, 64))
    diff = (model(x) - hf(x).logits).abs().max()
print(f"max |logit difference| = {diff:.2e}, max |logit| = {hf(x).logits.abs().max():.0f}")
# 124,318,464
# 124,439,808
# True
# max |logit difference| = 1.22e-04, max |logit| = 115
